In [ ]:
import asyncio
import hashlib
import json
from pathlib import Path

from IPython.display import display

import xyg
from test_graphforge import _base
from xyg import _native

native_path = Path(_native._lib._name).resolve()
assert native_path == Path(xyg.__file__).resolve().parents[2] / "target/release/libxyg_core.so"
native = {
    "path": str(native_path),
    "sha256": hashlib.sha256(native_path.read_bytes()).hexdigest(),
    "abi": int(_native._lib.xyg_abi_version()),
}
composition = xyg.compose_graphforge(base=_base("property"))
clicks, hovers, views, selections = [], [], [], []
chart = xyg.graphforge_chart(
    composition,
    title="Progressive property graph",
    on_click=clicks.append,
    on_hover=hovers.append,
    on_view_change=views.append,
    on_select=lambda selection: selections.append(selection.rows()),
)
widget = chart.show(display="widget")
assert chart._figure is None
display(widget)
other_clicks = []
other = xyg.graphforge_chart(
    composition, theme="dark", title="Independent graph", on_click=other_clicks.append
).widget()
display(other)

In [ ]:
assert asyncio.get_running_loop().is_running()
assert clicks and hovers and views, (clicks, hovers, views)
assert clicks[0]["id"] in composition.nodes.uuid
assert clicks[0]["graphforge"] == [composition.identify("node", 0)]
assert not other_clicks
print(
    "XYG graph kernel responsive",
    json.dumps(
        {
            "clicks": clicks,
            "native": native,
            "otherClicks": len(other_clicks),
            "hovers": len(hovers),
            "views": len(views),
            "selections": len(selections),
        }
    ),
)

In [ ]:
chosen_uuids = [composition.nodes.uuid[0], composition.edges.uuid[0]]
widget.select(uuids=chosen_uuids)
print("XYG graph Python rows selected")

In [ ]:
widget.reset_view()
print("XYG graph reset")

In [ ]:
final_pointer_before = len(clicks)
print("XYG graph pointer baseline", final_pointer_before)

In [ ]:
async with asyncio.timeout(5):
    while len(clicks) <= final_pointer_before:
        await asyncio.sleep(0.01)
assert clicks[-1]["id"] == composition.nodes.uuid[0]
assert clicks[-1]["graphforge"] == [composition.identify("node", 0)]
assert not other_clicks
widget.close()
other.close()
print(
    "XYG graph disposed",
    json.dumps(
        {
            "before": final_pointer_before,
            "after": len(clicks),
            "id": clicks[-1]["id"],
            "expectedId": composition.nodes.uuid[0],
        }
    ),
)